In [1]:
import os
import numpy as np
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

input_dir = "lemonleaf"
output_dir = "finaldataset"

os.makedirs(output_dir, exist_ok=True)

# Augmentation settings
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

MIN_TARGET = 900
MAX_TARGET = 1000

In [2]:
for class_name in os.listdir(input_dir):
    class_path = os.path.join(input_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    save_path = os.path.join(output_dir, class_name)
    os.makedirs(save_path, exist_ok=True)

    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    current_count = len(images)

    print(f"\nClass: {class_name} | Original: {current_count}")

    # Copy original images first
    for img_name in images:
        img = load_img(os.path.join(class_path, img_name), target_size=(224,224))
        img.save(os.path.join(save_path, f"orig_{img_name}"))

    # Decide target dynamically
    if current_count >= MIN_TARGET:
        target = current_count  # small augmentation or no need
    else:
        target = np.random.randint(MIN_TARGET, MAX_TARGET + 1)

    print(f"Target for {class_name}: {target}")

    count = current_count

    # AUGMENT ONLY IF REQUIRED
    while count < target:
        img_name = np.random.choice(images)
        img = load_img(os.path.join(class_path, img_name), target_size=(224,224))

        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)

        for batch in datagen.flow(x, batch_size=1):
            new_img = Image.fromarray(batch[0].astype(np.uint8))

            new_name = f"aug_{class_name}_{count}.jpg"
            new_img.save(os.path.join(save_path, new_name))

            count += 1
            break

    print(f"Final count: {class_name} -> {count}")


Class: Anthracnose | Original: 709
Target for Anthracnose: 960
Final count: Anthracnose -> 960

Class: Curl Virus | Original: 734
Target for Curl Virus: 908
Final count: Curl Virus -> 908

Class: Dry Leaf | Original: 100
Target for Dry Leaf: 988
Final count: Dry Leaf -> 988

Class: Healthy Leaf | Original: 730
Target for Healthy Leaf: 965
Final count: Healthy Leaf -> 965

Class: Spider Mites | Original: 100
Target for Spider Mites: 997
Final count: Spider Mites -> 997

Class: Citrus Canker | Original: 723
Target for Citrus Canker: 919
Final count: Citrus Canker -> 919

Class: Deficiency Leaf | Original: 730
Target for Deficiency Leaf: 925
Final count: Deficiency Leaf -> 925

Class: Sooty Mould | Original: 100
Target for Sooty Mould: 997
Final count: Sooty Mould -> 997

Class: Bacterial Blight | Original: 100
Target for Bacterial Blight: 985
Final count: Bacterial Blight -> 985


In [3]:
import os
import pandas as pd
from PIL import Image

dataset_path = "finaldataset"

data = []

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        image_count = 0
        sample_size = None

        for file in os.listdir(folder_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_count += 1

                if sample_size is None:
                    img = Image.open(os.path.join(folder_path, file))
                    sample_size = img.size

        data.append([folder, image_count, sample_size])

df = pd.DataFrame(data, columns=["Class", "Number of Images", "Pixel Size"])
df

,Class,Number of Images,Pixel Size
0,Anthracnose,960,"(224, 224)"
1,Bacterial Blight,985,"(224, 224)"
2,Citrus Canker,919,"(224, 224)"
3,Curl Virus,908,"(224, 224)"
4,Deficiency Leaf,925,"(224, 224)"
5,Dry Leaf,988,"(224, 224)"
6,Healthy Leaf,965,"(224, 224)"
7,Sooty Mould,997,"(224, 224)"
8,Spider Mites,997,"(224, 224)"


In [4]:
import os

dataset_path = "finaldataset"

total_images = 0

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        print(f"{class_name}: {count} images")
        total_images += count

print("\n====================")
print(f"TOTAL IMAGES: {total_images}")

Anthracnose: 960 images
Curl Virus: 908 images
Dry Leaf: 988 images
Healthy Leaf: 965 images
Spider Mites: 997 images
Citrus Canker: 919 images
Deficiency Leaf: 925 images
Sooty Mould: 997 images
Bacterial Blight: 985 images

TOTAL IMAGES: 8644


In [5]:
import os
from PIL import Image
import hashlib
dataset_path = "finaldataset"


corrupt_images = []
empty_files = []
hash_dict = {}
duplicates = []

# Function to get file hash (for duplicates)
def get_hash(file_path):
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)

    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

   
        # 1. Check empty file
        if os.path.getsize(img_path) == 0:
            empty_files.append(img_path)
            continue

   
        # 2. Check corrupt image
        try:
            img = Image.open(img_path)
            img.verify()
        except:
            corrupt_images.append(img_path)
            continue


        # 3. Check duplicates
        try:
            file_hash = get_hash(img_path)

            if file_hash in hash_dict:
                duplicates.append((img_path, hash_dict[file_hash]))
            else:
                hash_dict[file_hash] = img_path
        except:
            pass


# RESULTS

print("\n========== DATASET REPORT ==========\n")

print(f"Corrupt images   : {len(corrupt_images)}")
print(f"Empty files      : {len(empty_files)}")
print(f"Duplicate images : {len(duplicates)}")

print("\n---------- SAMPLE ISSUES ----------\n")

print("Corrupt examples:")
for img in corrupt_images[:5]:
    print(img)
print("\nEmpty file examples:")
for img in empty_files[:5]:
    print(img)
print("\nDuplicate examples:")
for dup in duplicates[:5]:
    print(dup)


print("CHECK COMPLETED ✔")


========== DATASET REPORT ==========

Corrupt images   : 0
Empty files      : 0
Duplicate images : 0

---------- SAMPLE ISSUES ----------

Corrupt examples:

Empty file examples:

Duplicate examples:
CHECK COMPLETED ✔
